# CTU-13 Botnet Detection — EDA (Scenario 1)

This is the second part of my NIDS Capstone. This time I wanted to work more on the actual EDA and understand the traffic before training any models.

I'm starting with **Scenario 1** and its bidirectional `.binetflow` file. The goal is to understand botnet, normal and background flows.

## 1. Loading the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_csv("https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/detailed-bidirectional-flow-labels/capture20110810.binetflow")

URLError: <urlopen error [Errno 110] Connection timed out>

In [ ]:
# The dataframe is huge, so pandas may hide the non-null counts by default.
# show_counts=True makes them visible.
df.info(show_counts=True)

In [ ]:
df.shape

In [ ]:
# First look at the actual flows and columns. One row is a network flow.
df.head()

## 2. Quick look at protocols and labels

In [ ]:
# Proto tells me the network/transport protocol, not the application (like HTTP or IRC).
df.Proto.unique()

In [ ]:
# UDP is everywhere in this capture. I still need to check whether it is mostly background traffic.
df.Proto.value_counts()

In [ ]:
df.Proto.value_counts().plot(kind='bar', x='Proto', y='count', figsize=(17,6), title="Almost all traffic is UDP traffic")

In [ ]:
# There are lots of missing values
df.isna().sum()

In [ ]:
# First I want to see how many different detailed labels we have.
df.Label.nunique()

In [ ]:
# The complete Label values describe more than just Botnet / Normal.
# Let's see the most frequent ones.
df.Label.value_counts().head(15)

## 3. Grouping the labels into traffic types

In [ ]:
# I want simpler groups for the EDA. Background is NOT the same as confirmed normal.
# For now I'll keep everything else as 'other'.
df['traffic_type'] = 'other'

In [ ]:
# Create broader categories from the original labels.
# I'm keeping the original Label column so I can check anything I'm unsure about.
df.loc[df.Label.str.contains('Background', na=False), 'traffic_type'] = 'Background'
df.loc[df.Label.str.contains('To-Normal', na=False), 'traffic_type'] = 'To-Normal'
df.loc[df.Label.str.contains('From-Normal', na=False), 'traffic_type'] = 'From-Normal'
df.loc[df.Label.str.contains('From-Botnet', na=False), 'traffic_type'] = 'From-Botnet'

In [ ]:
# The vast majority is Background. Interesting, but it isn't a verified normal class.
df.traffic_type.value_counts(normalize=True) * 100

In [ ]:
# What ended up in 'other'? Only nine Windows Update flows in this scenario.
# I'll leave them as 'other' for now instead of relabeling them.
df[df['traffic_type'] == 'other'].Label.value_counts()

In [ ]:
# The label distribution is very uneven. That's worth remembering for modelling.
df['traffic_type'].value_counts().plot(kind='bar', figsize=(10, 5))
plt.title('Distribution of network traffic categories')
plt.xlabel('Traffic type')
plt.ylabel('Number of flows')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Packets and bytes — a quick comparison

I'm comparing packet and byte counts. A few really large flows can push the **mean** up, so I also want to look at the **median**.

In [ ]:
df.groupby('traffic_type')[['TotPkts', 'TotBytes']].describe()

In [ ]:
df.groupby('traffic_type')[['SrcBytes', 'TotBytes']].describe()

A smaller table is easier to compare than the full `describe()` output:

In [ ]:
# Same information, but easier to read: only confirmed botnet and normal flows.
df.loc[
    df['traffic_type'].isin(['From-Botnet', 'From-Normal'])
].groupby('traffic_type')[['TotPkts', 'TotBytes', 'SrcBytes']].agg(['mean', 'median']).round(2)

## 5. Missing values — where are they coming from?

In [ ]:
# First check how much data is missing. No cleaning yet.
missing = pd.DataFrame({
    'missing': df.isna().sum(),
    'missing_pct': df.isna().mean() * 100
})
missing = missing[missing['missing'] > 0]
missing.sort_values('missing', ascending=False)

In [ ]:
missing_by_proto = df.groupby('Proto')['dTos'].agg(
    total = 'size',
    missing=lambda x:x.isna().sum()
)

missing_by_proto['missing_pct'] = (
    missing_by_proto['missing']/
    missing_by_proto['total'] * 100
)

missing_by_proto.sort_values(
    'total', ascending=False
).head(10)

In [ ]:
# Just looking at a few examples, some missing dTos values show up with -> / ?>
# and states like INT or A_. That's an observation, not a proof.
df.loc[
    df['dTos'].isna(),
    ['Proto', 'Dir', 'State', 'TotPkts', 'dTos']
].head(20)

In [ ]:
# Check ALL directions.
missing_by_dir = df.groupby('Dir')['dTos'].agg(
    total='size',
    present='count'
)
missing_by_dir['Missing'] = missing_by_dir['total'] - missing_by_dir['present']
missing_by_dir['pct'] = round(missing_by_dir['Missing'] / missing_by_dir['total'] * 100, 2)
missing_by_dir

In [ ]:
# dTos looks much more often missing in From-Botnet flows than From-Normal flows.
missing_by_type = df.groupby('traffic_type')['dTos'].agg(
    total='size',
    present='count'
)
missing_by_type['missing'] = missing_by_type['total'] - missing_by_type['present']
missing_by_type['pct'] = missing_by_type['missing'] / missing_by_type['total'] * 100
missing_by_type

In [ ]:
# Does this difference remain when I compare flows with the same direction?
# It does for -> : From-Botnet about 86.6% missing, From-Normal about 1.6%.
# That is an interesting clue
missing_td = df.groupby(['traffic_type', 'Dir'])['dTos'].agg(
    total='size',
    present='count'
)
missing_td['missing'] = missing_td['total'] - missing_td['present']
missing_td['pct'] = missing_td['missing'] / missing_td['total'] * 100
missing_td

**My takeaway:** For direction `->`, missing `dTos` is much more common in `From-Botnet` than `From-Normal`. The direction alone does not explain the whole difference. I still need to be careful: this could depend on the protocol, state or this specific capture.

## 6. Flow duration

In [ ]:
# Botnet flows have a lower mean duration than normal flows, but a higher median.
# Dur is measured in seconds, and we're talking about flows, not individual packets.
duration_stats = df.groupby('traffic_type')['Dur'].agg(
    ['count', 'mean', 'median', 'min', 'max']
)
duration_stats.round(3)

In [ ]:
duration_sample = df['Dur'].sample(
    n=20000,
    random_state=42
)

plt.figure(figsize=(10, 5))

plt.hist(
    np.log1p(duration_sample),
    bins=50
)

plt.title('Distribution of Network Flow Duration')
plt.xlabel('log(1 + Duration in seconds)')
plt.ylabel('Number of Flows')
plt.show()

In [ ]:
botnet_dur = df.loc[
    df['traffic_type'] == 'From-Botnet', 'Dur'
]

normal_dur = df.loc[
    df['traffic_type'] == 'From-Normal', 'Dur'
]

plt.figure(figsize=(10, 5))

plt.hist(
    np.log1p(botnet_dur.sample(
        n=min(5000, len(botnet_dur)),
        random_state=42
    )),
    bins=50,
    alpha=0.5,
    density=True,
    label='Botnet'
)

plt.hist(
    np.log1p(normal_dur.sample(
        n=min(5000, len(normal_dur)),
        random_state=42
    )),
    bins=50,
    alpha=0.5,
    density=True,
    label='Normal'
)

plt.xlabel('log(1 + Duration in seconds)')
plt.ylabel('Density')
plt.title('Flow Duration: Botnet vs Normal')
plt.legend()
plt.show()

## 7. Protocols vs. traffic types

In [ ]:
# Botnet and normal flows both use a lot of UDP and TCP.
# The raw counts alone don't tell which protocol is more suspicious.
protocol_counts = pd.crosstab(
    df['traffic_type'],
    df['Proto']
)
protocol_counts

In [ ]:
protocol_pct = pd.crosstab(
    df['traffic_type'],
    df['Proto'],
    normalize = 'index'
) * 100

protocol_pct[['tcp', 'udp', 'icmp']].round(2)

In [ ]:
protocol_pct.loc[
    ['From-Botnet', 'From-Normal', 'Background'],
    ['tcp', 'udp', 'icmp']
].plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title('Protocol Distribution by Traffic Category')
plt.xlabel('Traffic Category')
plt.ylabel('Percentage of Flows')
plt.xticks(rotation=0)
plt.legend(title='Protocol')
plt.tight_layout()
plt.show()

**What this shows me:** The percentages are for each traffic type, not the share of all TCP or UDP flows that are attacks. UDP dominates both labeled classes, so protocol alone does not look like a great correlation.

## 8. What I learned so far

I think this is a decent start. I didn't find one magical feature that separates botnet from normal traffic, but that's fine.

- Most flows are labeled **Background**. I can't automatically call them normal traffic.
- Botnet and normal flows both mostly use **UDP and TCP**, so just knowing the protocol probably won't be enough.
- The **mean and median duration** tell different stories. A few very long flows affect the averages.
- **`dTos` is missing a lot more often in From-Botnet flows**, even within direction `->`. It might be useful, but I still don't know if it's actual botnet behavior or something specific to this capture.
- Packet and byte values have big outliers, so I should not rely only on means.

### Before I start modelling

I want to check a smaller dataset containing just the clearly labeled **From-Botnet** and **From-Normal** flows. I'll keep **Background** out of the first supervised model because it isn't a confirmed normal class.

I still need to check duplicates, missing values, port formats and timestamps. I also don't want the model to simply memorize the infected machine's IP address, so **`Label`, `traffic_type`, `SrcAddr` and `DstAddr` should not be predictors**. I'll decide how to handle missing values after splitting the data, using only training data to fit any preprocessing.

## 9. Quick checks before modelling (not cleaning yet)

I'll start with only the clearly labeled flows. These checks tell me what needs attention before creating `X` and `y`. I'm **not** dropping or filling missing values here.

In [ ]:
labeled_flows = df.loc[
    df['traffic_type'].isin(['From-Botnet', 'From-Normal'])
].copy()

print('Labeled rows:', labeled_flows.shape[0])
print('Class counts:')
print(labeled_flows['traffic_type'].value_counts())
print('Exact duplicate rows:', labeled_flows.duplicated().sum())

In [ ]:
print('Missing values (%) in labeled flows:')
display((labeled_flows.isna().mean() * 100).sort_values(ascending=False).head(8).round(2))

print('Columns to inspect before modelling:')
print(labeled_flows[['StartTime', 'Sport', 'Dport']].dtypes)

print('Negative duration / packet / byte values:')
print((labeled_flows[['Dur', 'TotPkts', 'TotBytes', 'SrcBytes']] < 0).sum())
print('Flows with SrcBytes > TotBytes:',
      (labeled_flows['SrcBytes'] > labeled_flows['TotBytes']).sum())